## FRANCE
## Discover Feed AB Experiment
#### Overview:
- experiment details: JIRA TICKET
- experiment started on 4th March and ended on 24th April. Users were assigned to experiment till 13th April
- hypothesis: by showing Discover Feed in a new way (upranking content related to portfolio/watchlist, promoting investing plans, using tagged instruments from posts), we can increase the amount of investing plans created or trades made.
- experiment variants: 
    - [1] Control (50% of users): users were seeing existing, simplified Discover Feed described in JIRA TICKET
    - [2] Test (50% of users): users were seeing new, more tailored to them Discover Feed
- users were divided to two groups: old users who already traded before being assigned to experiment and new users, who did not trade before experiment started
- number of old users that were correctly qualified to experiment: 2499, users number in Control Group: 1232, users number in Test Group: 1267
- number of new users that were correctly qualified to experiment: 7175, users number in Control Group: 3599, users number in Test Group: 3576

#### Metrics:
- number of users who traded
- number of trades made
- number of created investment plans

#### Recommendation: keep Test variant
- Among new users there was significant difference between Test and Control groups in number of traders and number of trades made
- Among old users there was no significant difference between Test and Control groups in number of traders and number of trades made
- As for number of created investment plans, there was no significant difference between Test and Control groups both among old and new users. Main reason behind it is low number of created IPs, which impacts results. However it is worth to mention that longer the experiment lasted, p-value was going steadily down (which means that difference between Test and Control groups was growing) in favour of Test group (both among old and new users)

In [ ]:
# %pip install scipy
# %pip install --upgrade seaborn

In [ ]:
# import libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
import scipy.stats as stats
from itertools import product
import matplotlib.dates as mdates
%matplotlib inline

In [ ]:
# import data from .csv
df = pd.read_csv("discover_feed.csv")
df_old = df[df['traded_before_exp']==1]
df_new = df[df['traded_before_exp']==0]

In [ ]:
# check if data was imported correctly
# df_new.head()

### 2. General insights

#### 2.1. Creating functions

In [ ]:
def user_counts(df):

    # users in experiment
    users_count = df[["experiment_group", "user_id"]].groupby(["experiment_group"]).agg({"user_id":"nunique"}).reset_index()
    users_count.rename(columns={'user_id': 'users_count'}, inplace=True)
    # users_count.head()

    users_count_daily = df[["date_", "experiment_group", "user_id"]].groupby(["date_", "experiment_group"]).agg({"user_id":"nunique"}).reset_index()
    users_count_daily.rename(columns={'user_id': 'users_count'}, inplace=True)
    users_count_daily["date_text"] = pd.to_datetime(users_count_daily["date_"]).dt.strftime('%d/%m/%Y')
    users_count_daily.head()

    # display number of users in each experiment group
    plt.figure(figsize=(8,4))
    l1 = sns.barplot(data=users_count, x="experiment_group", y="users_count", hue="experiment_group")
    plt.ylabel("Users count")
    plt.xlabel("Experiment group")
    plt.title(f"Total number of users assigned to experiment")
    plt.show()

    # plot cumulative number of users assigned to experiment
    plt.figure(figsize=(16,4))
    l1 = sns.lineplot(data=users_count_daily, x="date_", y="users_count", hue="experiment_group")
    plt.ylabel("Users count")
    plt.xlabel("",)
    plt.title(f"Cumulative number of users assigned to experiment")
    plt.xticks(users_count_daily["date_"], users_count_daily["date_text"], rotation=90)
    plt.show()

In [ ]:
def metrics_charts(df):

    df_gr = df.groupby(["date_", "experiment_group"]).agg(\
                            {"orders_count":"sum",
                             "orders_amount":"sum",
                             "ip_count":"sum",
                             "is_trader":"sum"}).reset_index()

    df_gr.rename(columns={'is_trader': 'traders_count'}, inplace=True)
    df_gr["date_text"] = pd.to_datetime(df_gr["date_"]).dt.strftime('%d/%m/%Y')

    chart_metrics = ['traders_count', 'orders_count', 'orders_amount', 'ip_count']

    for metric in chart_metrics:
        plt.figure(figsize=(16,4))
        l1 = sns.lineplot(data=df_gr, x="date_", y=metric, hue="experiment_group")
        plt.ylabel(metric)
        plt.xlabel("",)
        plt.title(f"Cumulative number of {metric}")
        plt.xticks(df_gr["date_"], df_gr["date_text"], rotation=90)
        plt.show()

In [ ]:
def pvalue_calc(df):

    df = df.sort_values(by=['date_', 'user_id'])

    # list of days to iterate
    days_to_iterate = df["date_"].unique()
    # metrics to be evaluated
    metrics = ["is_trader", "orders_count", "ip_count"]

    # run Mann-Whitney-U test: one sided for each day of experiment
    dates_list = []
    metrics_list = []
    p_value_list = []
    test_value = []
    control_value = []
    for i_date, j_metric in product(days_to_iterate, metrics):
        test_group = df[(df["experiment_group"]== "Test") & (df["date_"] == i_date)]
        control_group = df[(df["experiment_group"]== "Control") & (df["date_"] == i_date)]
        results = stats.mannwhitneyu(x=test_group[j_metric], y=control_group[j_metric], alternative = "greater", method = "auto", nan_policy = "omit").pvalue
        dates_list.append(i_date)
        metrics_list.append(j_metric)
        p_value_list.append(results)

    # create empty df and save results
    daily_p_value = pd.DataFrame(columns=["date_", "metric", "p_value"])
    daily_p_value = pd.DataFrame(data={"date_":dates_list, "metric":metrics_list, "p_value":p_value_list})
    daily_p_value["date_text"] = pd.to_datetime(daily_p_value["date_"]).dt.strftime('%d/%m/%Y')

    # plot p-value for each metric

    for metric in metrics:

        plt.figure(figsize=(16,3))
        l1 = sns.lineplot(data=daily_p_value, x="date_", y=daily_p_value["p_value"][daily_p_value["metric"]==metric])
        plt.xlabel(None)
        plt.ylabel(None)
        plt.title(f"p-value: {metric}")
        plt.xticks(daily_p_value["date_"], daily_p_value["date_text"], rotation=90)
        plt.axhline(0.05, linestyle="-.", color = "red")
        l1.set_ylim(0,1.5)
        plt.show()

#### 2.2. Experiment overview and difference between test and control - trends

## Users who traded before experiment started

In [ ]:
user_counts(df_old)

In [ ]:
metrics_charts(df_old)

In [ ]:
pvalue_calc(df_old)

## New users who did not trade before experiment started

In [ ]:
user_counts(df_new)

In [ ]:
metrics_charts(df_new)

In [ ]:
pvalue_calc(df_new)